# Домашняя работа 1 — доли исходов HMDA

## Подготовка

In [ ]:
from pathlib import Path
import sys
import pandas as pd

DATA = Path("../data")
assert DATA.is_dir(), "Рядом с ноутбуком должна находиться папка data/"
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("Рабочая папка:", Path.cwd())

## Основной набор decisions.csv

In [ ]:
home_decisions = pd.read_csv(
    DATA / "decisions.csv",
    dtype={"lei": "string", "county_code": "string"},
)
home_outcomes = home_decisions["action_taken"]
home_n = len(home_decisions)

assert home_n == 6830
assert home_outcomes.notna().all()
assert home_outcomes.isin([1, 2, 3]).all()

print("Размер таблицы:", home_decisions.shape)
print(home_outcomes.value_counts().sort_index())

## H1. Доля отказов

In [ ]:
home_denied = int(home_outcomes.eq(3).sum())
home_approved = int(home_outcomes.isin([1, 2]).sum())
home_denial_rate = home_denied / home_n

print(f"Условие: action_taken == 3")
print(f"Числитель: {home_denied}")
print(f"Знаменатель: {home_n}")
print(f"Доля отказов: {100 * home_denial_rate:.2f}%")

assert home_denied + home_approved == home_n
assert 0 <= home_denial_rate <= 1

## H2. Одобрение без принятия предложения

In [ ]:
home_not_accepted = int(home_outcomes.eq(2).sum())
home_issued = int(home_outcomes.eq(1).sum())
home_not_accepted_rate = home_not_accepted / home_n
home_approval_rate = home_approved / home_n
home_issuance_rate = home_issued / home_n
difference_pp = (home_approval_rate - home_issuance_rate) * 100

print(f"Условие: action_taken == 2")
print(f"Числитель: {home_not_accepted}")
print(f"Знаменатель: {home_n}")
print(f"Одобрено, но не принято: {100 * home_not_accepted_rate:.2f}%")
print(f"Одобрено всего: {home_approved}")
print(f"Выдано: {home_issued}")
print(f"Разность долей одобрения и выдачи: {difference_pp:.2f} п.п.")

assert home_approved - home_issued == home_not_accepted
assert abs(difference_pp - 100 * home_not_accepted_rate) < 1e-10

## H3. Объяснение инструментов

- **Series** — одномерная структура pandas с индексом; **DataFrame** — двумерная таблица из строк и столбцов. Примеры: `home_decisions["action_taken"]` и `home_decisions`.
- **nrows** — параметр `read_csv`, ограничивающий количество строк при чтении файла. **head()** — метод уже загруженной таблицы, показывающий первые строки.
- **Метод** выполняет действие и вызывается со скобками, например `home_outcomes.eq(3)`. **Атрибут** возвращает свойство объекта, например `home_decisions.shape`.

Документация: [pandas.Series.isin](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html). Метод проверяет принадлежность каждого элемента набору значений и возвращает булеву Series.

## H+1. Другой набор записей

In [ ]:
raw = pd.read_csv(DATA / "hmda_2023_dc_raw.csv")
raw_outcomes = raw["action_taken"]
raw_n = len(raw)
raw_issued = int(raw_outcomes.eq(1).sum())
raw_issuance_rate = raw_issued / raw_n

home_issued = int(home_outcomes.eq(1).sum())
home_issuance_rate = home_issued / home_n

print(f"raw: {raw_issued}/{raw_n} = {100 * raw_issuance_rate:.2f}%")
print(f"decisions: {home_issued}/{home_n} = {100 * home_issuance_rate:.2f}%")
print("Исходы в raw:")
print(raw_outcomes.value_counts().sort_index())
assert raw_issued == home_issued
assert raw_n >= home_n

## H+2. Тип сравниваемого значения

In [ ]:
text_example = pd.read_csv(
    DATA / "reading_example.csv",
    dtype={"action_taken": "string"},
)
text_actions = text_example["action_taken"]

eq_number = text_actions.eq(1)
eq_text = text_actions.eq("1")
isin_numbers = text_actions.isin([1, 2])
isin_texts = text_actions.isin(["1", "2"])

print("eq(1):", int(eq_number.sum()))
print('eq("1"):', int(eq_text.sum()))
print("isin([1, 2]):", int(isin_numbers.sum()))
print('isin(["1", "2"]):', int(isin_texts.sum()))

assert int(eq_number.sum()) == 0
assert int(eq_text.sum()) == 2
assert int(isin_numbers.sum()) == 0
assert int(isin_texts.sum()) == 4